## Steps


# Univariate Fine-Mapping and TWAS with SuSiE

## Description

This pipeline performs per-region univariate fine-mapping with SuSiE and computes TWAS
weights, via the `susie_twas` workflow of `pipeline/mnm_regression.ipynb`. For each
region the workflow fits SuSiE on the covariate-residualized phenotype × genotype
matrix and then cross-validates five prediction models (**enet**, **lasso**,
**bayes_r**, **mrash**, **susie**) to pick TWAS weights.

SuSiE searches for up to five independent signals per region; if all five are found,
the cap is raised. SuSiE itself does not accept covariates, so covariates are regressed
out of both the phenotype and the genotype matrix beforehand.

### What is a "region"?

A region is **one row in the phenotype manifest** (`--phenoFile`) plus its association
window (from `--customized-association-windows`, or a symmetric `--cis-window` around
`start/end`). One `susie_twas` call fits one SuSiE + TWAS model per region.

* **Gene-based phenotypes.** Manifest row = one gene; window = gene's containing TAD (or
  ± cis-window around the TSS).
* **Peak-based phenotypes (caQTL, mQTL).** Manifest row = one peak/CpG; window = the
  peak's containing [extended-TAD](./../reference_data/generalized_TADB.html). A peak is
  2–5 kb and cannot serve as its own window.


## Input Files

| File | Description |
| --- | --- |
| `<out_dir>/pheno_manifest.tsv` | Phenotype manifest mapping each region to its phenotype/covariate files |
| `<out_dir>/association_windows.bed` | Association-test windows (one row per region) |
| `<out_dir>/peaks_split/<ID>.bed.gz{,.tbi}` | Per-region phenotype data, bgzipped and tabix-indexed |


### Step 1 -- Prepare the phenotype manifest and association-windows BED

The pipeline expects two derived files per experiment:

* `pheno_manifest.tsv` — 5 columns (`#chr start end ID path`), one row per region.
  `path` points to a bgzipped+tabixed phenotype BED restricted to that region.
* `association_windows.bed` — 4 columns (`chr start end ID`), one row per region.
  `start/end` define the cis search window; `ID` must match a row in the manifest.

For gene-based phenotypes these files are produced by the standard phenotype
preprocessing + [TADB generation](./../reference_data/generalized_TADB.html) steps.

For peak-based phenotypes (caQTL, mQTL) you can derive both from the raw peak BED with
the helper below; it renames the peak-ID header, splits the multi-sample BED into one
bgz+tbi per peak, writes the manifest, and annotates each peak with its smallest
containing extended-TAD.

In [ ]:
bash code/misc/preprocess/prepare_peak_inputs.sh \
    <raw_peaks.bed.gz> \
    <extended_TADB.bed> \
    <out_dir>

# Produces:
#   <out_dir>/pheno_manifest.tsv
#   <out_dir>/association_windows.bed
#   <out_dir>/peaks_split/<ID>.bed.gz{,.tbi}


### Step 2 -- Run `susie_twas`

**Gene-based example** (3 genes, default bulk-RNA-seq preprocessing):

In [ ]:
# Step 1: build QtlDataset RDS (merge genotype, phenotype, covariates)
sos run pipeline/qtl_dataset.ipynb qtl_dataset_construct \
    --cwd output/qtl_dataset \
    --study protocol_example \
    --genotype-prefix input/colocboost/example.chr22 \
    --phenotype-manifest input/colocboost/pheno_manifest_multicontext.tsv \
    --covariate-file input/colocboost/example_covariates.tsv \
    --customized-association-windows input/colocboost/association_windows.bed \
    --transpose-covariates

# Step 2: SuSiE fine-mapping per region
sos run pipeline/fine_mapping.ipynb fine_mapping \
    --cwd output/fine_mapping \
    --study protocol_example \
    --qtl-dataset output/qtl_dataset/protocol_example.qtl_dataset.rds \
    --region-name ENSG00000130538

# Step 3: TWAS weights (10 methods + SR-TWAS ensemble)
sos run pipeline/twas_weights.ipynb twas_weights \
    --cwd output/twas_weights \
    --study protocol_example \
    --qtl-dataset output/qtl_dataset/protocol_example.qtl_dataset.rds \
    --fine-mapping-result output/fine_mapping/protocol_example.ENSG00000130538.finemap.rds \
    --region-name ENSG00000130538

**Peak-based example** (caQTL; 10 chr22 peaks, 1,287 samples):

In [ ]:
# Smaller example: same chain, different data.
sos run pipeline/qtl_dataset.ipynb qtl_dataset_construct \
    --cwd output/qtl_dataset \
    --study example_10peaks_twas \
    --genotype-prefix input/example.chr22 \
    --phenotype-manifest output/inputs_ready/pheno_manifest.tsv

sos run pipeline/fine_mapping.ipynb fine_mapping \
    --cwd output/fine_mapping \
    --study example_10peaks_twas \
    --qtl-dataset output/qtl_dataset/example_10peaks_twas.qtl_dataset.rds \
    --regions chr22:15000000-17000000

sos run pipeline/twas_weights.ipynb twas_weights \
    --cwd output/twas_weights \
    --study example_10peaks_twas \
    --qtl-dataset output/qtl_dataset/example_10peaks_twas.qtl_dataset.rds \
    --regions chr22:15000000-17000000

Expected log (both cases):

```
INFO: Running get_analysis_regions:
Loading customized association analysis window from <windows_bed>
INFO: get_analysis_regions is completed.
INFO: get_analysis_regions output:   regional_data
INFO: Running susie_twas:
INFO: susie_twas (index=0) is completed.
INFO: susie_twas (index=1) is completed.
...
INFO: susie_twas output:   <cwd>/fine_mapping/<name>.<chrom>_<ID>.univariate_bvsr.rds
                           <cwd>/twas_weights/<name>.<chrom>_<ID>.univariate_twas_weights.rds
                           ...  (2 × N_regions items)
INFO: Workflow susie_twas is executed successfully.
```


## Command Interface

In [ ]:
sos run pipeline/fine_mapping.ipynb -h
sos run pipeline/twas_weights.ipynb -h

Flags consumed by `susie_twas`:

* `--genoFile` — path to a PLINK1 `.bed` file (pass the `.bed` path, not the prefix).
  FID/IID in the `.fam` must match the phenotype/covariate sample IDs.
* `--phenoFile` — 5-column `#chr start end ID path` manifest. One row per region; `path`
  points at a bgzipped+tabixed phenotype BED restricted to that region. Pass multiple
  files for multi-condition analyses.
* `--covFile` — tab-delimited, literal `ID` header in column 1; covariate names in rows,
  sample IDs across the header row.
* `--customized-association-windows` — 4-column `chr start end ID` BED. Required unless
  `--cis-window` is non-negative.
* `--cis-window` — symmetric cis radius (bp) around each region's `start/end`. Default
  `-1` (forces the customized-windows path).
* `--phenotype-names` — condition name(s), one per `--phenoFile`.
* `--max-cv-variants` — cap on variants used in cross-validation (default 5,000).
* `--ld_reference_meta_file` — summary-stats / RSS mode only; omit for individual-level.
* `--region-name` — restrict to one or more region IDs for sanity checks.
* `--save-data` — also emit `*.univariate_data.rds` with the residualized X/Y matrices.


## Output Format

All outputs are written under `--cwd`, one set per region:

**Fine-mapping** (`output/fine_mapping/`):
- `{study}.{region_id}.finemap.rds` — `QtlFineMappingResult` S4 object, one entry per context

**TWAS weights** (`output/twas_weights/`):
- `{study}.{region_id}.twas_weights.rds` — `TwasWeights` S4 object

### `TwasWeights` S4 structure

Top-level via `attr(tw, "listData")`:

| Field | Content |
|-------|------|
| `study` | Study name string |
| `context` | Context name per entry (repeated) |
| `trait` | Gene/trait ID per entry |
| `method` | Method name per entry: `mrash`, `susie`, `susie_inf`, `enet`, `lasso`, `mcp`, `scad`, `l0learn`, `bayes_r`, `bayes_c`, `ensemble` |
| `entry` | List of `TwasWeightsEntry` S4 objects (one per context × method) |

Total entries = N_context × 11 methods. For 2 contexts: 22 entries.

### `TwasWeightsEntry` attributes

Each entry accessed via `attr(entry, "<name>")`:

| Attribute | Class | Description |
|-----------|-------|-------------|
| `variantIds` | character | SNP IDs (`chr:pos:ref:alt`), length = number of variants in region |
| `weights` | numeric | Per-SNP TWAS weights (β from genotype→expression model) |
| `fits` | model object | Fitted model (class: `mr.ash`, `susie`, `glmnet`, etc.) |
| `cvPerformance` | list | CV results: `samplePartition`, `predictions`, `metrics` |
| `standardized` | logical | Whether X/Y were standardized (default FALSE) |

`cvPerformance$metrics` (for non-ensemble methods):
```
corr      rsq   adj_rsq      pval      RMSE       MAE
```

`cvPerformance` for ensemble entry:
- `methodCoef`: named numeric — ζ coefficient for each method (non-negative, sum to 1)
- `methodPerformance`: named numeric — per-method CV R²


## Anticipated Results

Quick sanity check — inspect cross-validation performance for the first region:

### Fine-Mapping Result: `QtlFineMappingResult` S4

The fine-mapping RDS holds **all contexts for one gene** in a single object.
Access the structured data via `attr(rds, "listData")`:

| Field | Description |
|-------|-------------|
| `study` | Study name string (from `--study`) |
| `context` | Context label for each entry (one entry per context) |
| `trait` | Gene/trait ID (Ensembl) |
| `method` | Fine-mapping method used (`"susie"`) |
| `entry` | List of `FineMappingEntry` S4 objects, one per context |

Each `FineMappingEntry` has these attributes:

| Attribute | Class | Description |
|-----------|-------|-------------|
| `variantIds` | character | SNP IDs in the region (`chr:pos:ref:alt`) |
| `susieFit` | susie | Raw `susie()` output — contains `sets$cs` (credible sets), `lbf` (log Bayes factors), `alpha` (posterior inclusion probabilities per effect) |
| `topLoci` | data.frame | Per-variant summary table (see column reference below) |
| `cvResult` | list | Cross-validation result used to compute TWAS weights |

**`topLoci` column reference** (1024 rows × 23 columns for this example):

| Column | Description |
|--------|-------------|
| `variant_id` | SNP ID (`chr:pos:ref:alt`) |
| `chrom`, `pos` | Chromosome and position |
| `A1`, `A2` | Effect and reference alleles |
| `N` | Sample size |
| `af` | Allele frequency of A1 |
| `marginal_beta`, `marginal_se` | Marginal effect estimate and standard error |
| `marginal_z`, `marginal_p` | Marginal z-score and p-value |
| `pip` | **Posterior Inclusion Probability** — probability this variant is causal (0–1) |
| `posterior_mean`, `posterior_sd` | Posterior effect size estimate and uncertainty |
| `cs_95`, `cs_70`, `cs_50` | Which credible set the variant belongs to at each coverage level; empty string = not in any CS |
| `cs_95_purity` | Min pairwise LD r² within the 95% CS; **≥ 0.5 required** for a credible CS |
| `method` | Fine-mapping method (`"susie"`) |
| `gene`, `event` | Gene ID and molecular event type |
| `grange_start`, `grange_end` | Association window boundaries |


In [ ]:
suppressPackageStartupMessages(library(pecotmr))

# Load QtlFineMappingResult
rds <- readRDS("output/fine_mapping/protocol_example.ENSG00000130538.finemap.rds")
class(rds)   # "QtlFineMappingResult"

ld <- attr(rds, "listData")
ld$context   # "context1"  "context2"
ld$method    # "susie"  "susie"

# --- Step 1: check for valid credible sets ---
e1 <- ld$entry[[1]]          # context1
sf <- attr(e1, "susieFit")
sf$sets$cs                   # NULL → no credible set passed purity filter
sf$lbf                       # 0.027 0.027 0.027 0.028 0.028 → all near zero

# --- Step 2: top variants by PIP ---
tl <- attr(e1, "topLoci")
tl_s <- tl[order(tl$pip, decreasing = TRUE), ]
head(tl_s[, c("variant_id", "af", "marginal_z", "marginal_p", "pip", "cs_95_purity")], 3)
#            variant_id     af  marginal_z  marginal_p     pip  cs_95_purity
# chr22:16445263:A:T  0.837    3.482    4.99e-04   0.032            0
# chr22:16418860:A:T  0.704   -3.436    5.91e-04   0.031            0
# chr22:16214125:A:T  0.612    3.307    9.42e-04   0.027            0

# --- Interpretation ---
# cs_95_purity = 0 on all variants → the single CS covers the whole region
# (toy data, n=49: insufficient power to fine-map)
# In real data (n ≥ 500): expect PIP > 0.5 lead variants
# and cs_95_purity ≥ 0.5 credible sets of 1–10 SNPs


### TWAS Weights Result: `TwasWeights` S4

The TWAS weights RDS holds weights from **all methods × all contexts** for one gene.
Top-level structure via `attr(tw, "listData")`:

| Field | Description |
|-------|-------------|
| `study` | Study name |
| `context` | Context label for each entry (repeated per method) |
| `trait` | Gene/trait ID |
| `method` | Method name: `mrash`, `susie`, `susie_inf`, `enet`, `lasso`, `mcp`, `scad`, `l0learn`, `bayes_r`, `bayes_c`, `ensemble` |
| `entry` | List of `TwasWeightsEntry` S4 objects (N_contexts × 11 methods total) |

Each `TwasWeightsEntry` is accessed via `attr(entry, "<name>")` — **not** `$` notation:

| Attribute | Description |
|-----------|-------------|
| `variantIds` | SNP IDs (`chr:pos:ref:alt`) for all variants in the region |
| `weights` | **Per-SNP TWAS weights** — the β vector used in Z_TWAS = **w**ᵀ**z** / √(**w**ᵀR**w**) |
| `fits` | Fitted model object (class-specific: `mr.ash`, `susie`, `glmnet`, etc.) |
| `cvPerformance` | CV results — for per-method entries: `$metrics` with `corr`, `rsq`, `adj_rsq`, `pval`, `RMSE`, `MAE`; for ensemble entry: `$methodCoef` (ζ per method) and `$methodPerformance` (CV R² per method) |
| `standardized` | Whether X/Y were z-score standardized before fitting (default: `FALSE`) |

**CV performance metrics** tell you how well each method predicts expression using out-of-fold cross-validation:
- `rsq` / `adj_rsq`: cross-validated R² — the primary metric for method quality
- `corr`: Pearson correlation between predicted and observed expression
- `pval`: p-value of the cross-validated prediction

**Ensemble `methodCoef`** (ζ): non-negative weights summing to 1 that determine how much each method contributes to the final ensemble prediction. `w_ensemble = Σ ζ_k × w_k`.


In [ ]:
suppressPackageStartupMessages(library(pecotmr))

tw  <- readRDS("output/twas_weights/protocol_example.ENSG00000130538.twas_weights.rds")
class(tw)     # "TwasWeights"

ld  <- attr(tw, "listData")
unique(ld$context)   # "context1"  "context2"
unique(ld$method)    # "mrash" "susie" "susie_inf" "enet" "lasso" "mcp" "scad" "l0learn" "bayes_r" "bayes_c" "ensemble"
length(ld$entry)     # 22

# Inspect one entry (context1 / susie)
e_susie <- ld$entry[ld$context == "context1" & ld$method == "susie"][[1]]
wt <- attr(e_susie, "weights")
cat("non-zero weights:", sum(wt != 0), "/", length(wt), "\n")

m <- attr(e_susie, "cvPerformance")$metrics
round(m[c("corr","rsq","adj_rsq","pval")], 4)
#    corr     rsq adj_rsq    pval
#  0.9798  0.9599  0.9591  0.0000

# Method summary for context1
idx1 <- which(ld$context == "context1")
for (i in idx1) {
  e <- ld$entry[[i]]
  m <- tryCatch(attr(e,"cvPerformance")$metrics, error=function(e) NULL)
  rsq <- if (!is.null(m) && is.numeric(m)) round(m["rsq"], 3) else NA
  cat(sprintf("%-12s  rsq=%s\n", ld$method[i], rsq))
}
# mrash         rsq=0.466
# susie         rsq=0.960
# susie_inf     rsq=0.305
# enet          rsq=0.060
# lasso         rsq=0.160
# mcp           rsq=NA
# scad          rsq=0.005
# l0learn       rsq=0.031
# bayes_r       rsq=0.520
# bayes_c       rsq=0.577
# ensemble      rsq=NA

# Ensemble coefficients
ens <- ld$entry[ld$context == "context1" & ld$method == "ensemble"][[1]]
cv  <- attr(ens, "cvPerformance")
round(cv$methodCoef[cv$methodCoef > 0.001], 3)   # bayes_r=0.802, bayes_c=0.198
round(cv$methodPerformance, 3)

Interpretation:

* `twas_weights$<method>` — use for TWAS (`Z ≈ weights · LD · GWAS z-scores`).
* `twas_cv_result$performance` — pick the method with highest `adj_rsq` before applying.
* `top_loci` in `univariate_bvsr.rds` — lead variants and credible-set coverages.
* `susie_result_trimmed$sets$cs` — indices of variants in each 95% credible set.

### Common pitfalls

* **Sample-ID drift.** The phenotype BED's header columns (5+), the bfile FID/IID, and
  the covariate TSV header must all be drawn from the same set. Sos does not warn on
  missing samples — it silently drops them and may report zero overlap.
* **Empty regions.** If a region has no variants in its association window (e.g. the
  manifest points at a window with no bfile coverage) sos marks it completed with an
  empty output. Verify the expected file count in the `INFO: susie_twas output:` line.
* **`--cis-window` vs `--customized-association-windows`.** Either/or. If both are
  passed the customized windows win. Pass `--cis-window -1` to force the customized
  path explicitly.
